In [ ]:
!pip install playwright
!playwright install chromium
!python scraper.py

!apt-get update
!apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixed3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libpangocairo-1.0-0 libcairo2 libnss3 libnspr4

!playwright install-deps chromium
!playwright install chromium

python3: can't open file '/content/scraper.py': [Errno 2] No such file or directory
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,122 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.s

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import csv
import re

MAPS_URL = "https://www.google.com/maps/search/dentist+mumbai/@19.233684,72.7041857,12z/data=!4m2!2m1!6e1?entry=ttu"

async def scrape_google_maps(url=MAPS_URL, max_results=100, get_contact_details=False):
    results = []
    seen_names = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(locale="en-US")

        # explicit timeout so goto can't hang forever - fails fast instead
        try:
            await page.goto(url, timeout=30000, wait_until="domcontentloaded")
        except Exception as e:
            print(f"Failed to load page: {e}")
            await browser.close()
            return results

        await page.wait_for_timeout(4000)

        try:
            await page.locator('button:has-text("Accept all")').click(timeout=3000)
        except Exception:
            pass

        # confirm the feed actually exists before looping - fail fast if not
        feed = page.locator('div[role="feed"]')
        try:
            await feed.wait_for(timeout=15000)
        except Exception:
            print("Results feed never loaded - Google likely blocked or changed layout.")
            await browser.close()
            return results

        last_count = 0
        stall_rounds = 0
        max_scroll_rounds = 30  # hard ceiling so this loop CANNOT run forever
        scroll_round = 0

        while len(results) < max_results and stall_rounds < 6 and scroll_round < max_scroll_rounds:
            scroll_round += 1
            cards = feed.locator('div.Nv2PK')
            count = await cards.count()

            for i in range(count):
                card = cards.nth(i)
                try:
                    name = await card.locator('div.qBF1Pd').inner_text(timeout=5000)
                except Exception:
                    continue

                if name in seen_names:
                    continue
                seen_names.add(name)

                try:
                    rating = await card.locator('span.MW4etd').inner_text(timeout=3000)
                except Exception:
                    rating = ""

                try:
                    reviews_raw = await card.locator('span.UY7F9').inner_text(timeout=3000)
                    reviews = re.sub(r"[()]", "", reviews_raw)
                except Exception:
                    reviews = ""

                try:
                    detail_line = await card.locator('div.W4Efsd').first.inner_text(timeout=3000)
                    detail_line = detail_line.replace("\n", " | ")
                except Exception:
                    detail_line = ""

                row = {"name": name, "rating": rating, "reviews": reviews, "details": detail_line}

                if get_contact_details:
                    try:
                        await card.click(timeout=5000)
                        await page.wait_for_timeout(1500)
                        try:
                            row["address"] = await page.locator('button[data-item-id="address"]').first.inner_text(timeout=3000)
                        except Exception:
                            row["address"] = ""
                        try:
                            row["phone"] = await page.locator('button[data-item-id^="phone"]').first.inner_text(timeout=3000)
                        except Exception:
                            row["phone"] = ""
                        try:
                            row["website"] = await page.locator('a[data-item-id="authority"]').first.inner_text(timeout=3000)
                        except Exception:
                            row["website"] = ""
                    except Exception:
                        pass

                results.append(row)
                print(f"Collected: {len(results)}/{max_results}")  # progress feedback
                if len(results) >= max_results:
                    break

            try:
                await feed.evaluate("el => el.scrollBy(0, 2000)")
            except Exception:
                break
            await page.wait_for_timeout(2000)

            if count == last_count:
                stall_rounds += 1
            else:
                stall_rounds = 0
            last_count = count

        await browser.close()

    return results


def save_csv(data, filename="mumbai_dentists.csv"):
    if not data:
        print("No results found.")
        return
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
    print(f"Saved {len(data)} results to {filename}")


data = await scrape_google_maps(get_contact_details=True, max_results=100)
save_csv(data)

Collected: 1/100
Collected: 2/100
Collected: 3/100
Collected: 4/100
Collected: 5/100
Collected: 6/100
Collected: 7/100
Collected: 8/100
Collected: 9/100
Collected: 10/100
Collected: 11/100
Collected: 12/100
Collected: 13/100
Collected: 14/100
Collected: 15/100
Collected: 16/100
Collected: 17/100
Collected: 18/100
Collected: 19/100
Collected: 20/100
Collected: 21/100
Collected: 22/100
Collected: 23/100
Collected: 24/100
Collected: 25/100
Collected: 26/100
Collected: 27/100
Collected: 28/100
Collected: 29/100
Collected: 30/100
Collected: 31/100
Collected: 32/100
Collected: 33/100
Collected: 34/100
Collected: 35/100
Collected: 36/100
Collected: 37/100
Collected: 38/100
Collected: 39/100
Collected: 40/100
Collected: 41/100
Collected: 42/100
Collected: 43/100
Collected: 44/100
Collected: 45/100
Collected: 46/100
Collected: 47/100
Collected: 48/100
Collected: 49/100
Collected: 50/100
Collected: 51/100
Collected: 52/100
Collected: 53/100
Collected: 54/100
Collected: 55/100
Collected: 56/100
C

In [ ]:
import os

print(os.getcwd())
print(os.listdir("/content"))

/content
['.config', 'mumbai_dentists.csv', 'sample_data']


In [21]:
import pandas as pd

df = pd.read_csv("mumbai_dentists.csv")
df

,name,rating,reviews,details,address,phone,website
0,Bombay Dental Centre,5.0,NaN,5.0,"1st floor, kamdenu Departmental store, Hubtow...",+91 89895 38512,bombaydentalcentre.com
1,"My Smile Dental Clinic Andheri West, Mumbai",4.9,NaN,4.9,"\nShop No.59, 1st Floor, Oshiwara Link Plaza ...",\n+91 98199 31333,\nmysmiledentalclinic.com
2,Make Me Smile Dental Clinic,5.0,NaN,5.0,"\nShop No.59, 1st Floor, Oshiwara Link Plaza ...",\n+91 98199 31333,\nmysmiledentalclinic.com
3,The Dental Bond,4.9,NaN,4.9,"\nA Wing, Samartha Aishwarya, 1006, Lokhandwa...",\n+91 98927 81987,\nthedentalbond.com
4,Dr. Shivani’s Dental Clinic,4.9,NaN,4.9,"\nAshtavinayak Towers, Moongipa Arcade, G-215...",\n+91 84465 69049,\ndrshivanikdentalclinic.com
...,...,...,...,...,...,...,...
95,Dr. Twinkle Sanghavi's Dental Clinic,4.8,NaN,4.8,"\nShop no. 5, OM SAINATH CHSL, Ram Mandir Rd,...",\n+91 77460 39006,NaN
96,Tiny Tooth Dentistry,4.9,NaN,4.9,"\nParijat building , ground floor, 10 vithal ...",\n+91 98206 74149,\ndrtwinklesanghavi.com
97,"Nova Smile Dental Clinic, नोवा स्माईल डेंटल क्...",4.5,NaN,4.5,"\n17, Rajnigandha Shopping Centre, opposite G...",\n+91 98923 82636,\ntinytooth.in
98,Aura Dental Care – Best Dentist in Andheri West,5.0,NaN,5.0,"\nUnit no 17, Silver streak, Mandir Masjid Ju...",\n+91 90229 04554,\nnovasmiledental.in


In [22]:
df = df.replace(r"[\n]", "", regex=True)

In [24]:
df["phone"] = df["phone"].astype(str).str.strip()
df["phone"] = "'" + df["phone"]

In [25]:
df = df.drop(columns=["reviews"])

In [26]:
df.to_excel("Clinic Lead.xlsx", index=False)

In [27]:
from google.colab import files

files.download("/content/Clinic Lead.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>